In [14]:
from google.cloud import texttospeech
from google.oauth2 import service_account

In [15]:
import re
from pathlib import Path

# Google Cloud TTS Pricing per million characters (USD)
RATES = {
    "Standard / Journey / WaveNet": 0.000004,  # $4 per 1M characters
    "Neural2 / Polyglot": 0.000016,          # $16 per 1M characters
    "Chirp 3 HD": 0.000030,                  # $30 per 1M characters
    "Studio": 0.000160,                      # $160 per 1M characters
}

def clean_markdown(text: str) -> str:
    """Removes common Markdown syntax so cost is calculated on spoken text only."""
    text = re.sub(r'```[\s\S]*?```', '', text)       # Remove code blocks
    text = re.sub(r'`[^`]*`', '', text)               # Remove inline code
    text = re.sub(r'!\[.*?\]\(.*?\)', '', text)       # Remove images
    text = re.sub(r'\[(.*?)\]\(.*?\)', r'\1', text)   # Keep link text, drop URL
    text = re.sub(r'[#*_\-~>`]', '', text)            # Remove Markdown styling characters
    return text.strip()

def estimate_tts_cost(file_path: str):
    path = Path(file_path)
    if not path.exists():
        print(f"Error: File '{file_path}' not found.")
        return

    raw_text = path.read_text(encoding="utf-8")
    cleaned_text = clean_markdown(raw_text)
    
    char_count = len(cleaned_text)
    word_count = len(cleaned_text.split())

    print(f"File: {path.name}")
    print(f"Word Count: ~{word_count:,}")
    print(f"Billable Characters: {char_count:,}\n")
    print("Estimated Cost breakdown (excl. free monthly tier):")
    print("-" * 50)
    
    for tier, rate in RATES.items():
        cost = char_count * rate
        print(f"{tier:<30} ${cost:.6f}")



In [16]:
# Example Usage
estimate_tts_cost("../module_1.md")

File: module_1.md
Word Count: ~4,184
Billable Characters: 26,479

Estimated Cost breakdown (excl. free monthly tier):
--------------------------------------------------
Standard / Journey / WaveNet   $0.105916
Neural2 / Polyglot             $0.423664
Chirp 3 HD                     $0.794370
Studio                         $4.236640


In [17]:
# Path to your JSON credentials file
KEY_PATH = "service_account.json"

# Load credentials from the JSON file
credentials = service_account.Credentials.from_service_account_file(KEY_PATH)

# Pass credentials directly to the client
client = texttospeech.TextToSpeechClient(credentials=credentials)

In [18]:
# 1. Read the Markdown file content
file_path = "../module_1.md"
text_content = Path(file_path).read_text(encoding="utf-8")

# 2. Pass the file content into SynthesisInput
synthesis_input = texttospeech.SynthesisInput(text=text_content)

In [19]:
synthesis_input


text: "\n## Architecturing basics\nDecomposition is where you assign each part of the request to Claude, to an existing system, or to a human, using the four properties of generative AI as the lens. Getting this wrong by over-assigning to Claude is the most common and most expensive early mistake.\nPattern selection is where you decide whether the work is an augmented call, a workflow, or an agent. Each choice provides and costs you something, naming the costs is the objective.\nReference architectures are where a known, good blueprint either fits the problem shape or is misapplied. The failure to watch for is retrieval quietly doing a job that the live transactional state should own.\nModel, context, and entry point are where you choose a model tier, a context strategy, and a delivery route, and where evaluations become a stage-gate before any model swap. Check if governance and regulated-industry constraints rule-out a route before considering any cost or latency tradeoffs.\n\nWhen y

In [20]:
voice = texttospeech.VoiceSelectionParams(
    language_code="en-US",
    name="en-US-Journey-F"
)

audio_config = texttospeech.AudioConfig(
    audio_encoding=texttospeech.AudioEncoding.MP3
)


In [21]:
# 1. Read file
file_path = "../module_1.md"
raw_text = Path(file_path).read_text(encoding="utf-8")

# 2. Split text into chunks under 4,000 chars (by paragraph/newline to keep natural pauses)
paragraphs = raw_text.split("\n\n")
chunks = []
current_chunk = ""

for p in paragraphs:
    if len(current_chunk) + len(p) < 4000:
        current_chunk += p + "\n\n"
    else:
        chunks.append(current_chunk)
        current_chunk = p + "\n\n"
if current_chunk:
    chunks.append(current_chunk)

# 3. Synthesize each chunk and append audio bytes
full_audio = bytearray()

for i, chunk in enumerate(chunks, 1):
    print(f"Synthesizing chunk {i}/{len(chunks)} ({len(chunk)} chars)...")
    synthesis_input = texttospeech.SynthesisInput(text=chunk)
    
    response = client.synthesize_speech(
        input=synthesis_input, voice=voice, audio_config=audio_config
    )
    full_audio.extend(response.audio_content)

# 4. Save combined MP3 file
with open("module_1.mp3", "wb") as out:
    out.write(full_audio)

print("Saved combined audio to module_1.mp3")

Synthesizing chunk 1/7 (3794 chars)...
Synthesizing chunk 2/7 (3865 chars)...
Synthesizing chunk 3/7 (3746 chars)...
Synthesizing chunk 4/7 (3894 chars)...
Synthesizing chunk 5/7 (3922 chars)...
Synthesizing chunk 6/7 (3921 chars)...
Synthesizing chunk 7/7 (3592 chars)...
Saved combined audio to module_1.mp3
